# LAB: Agents that Speak MCP Fluently

## Objective

In `mcp_introduction.ipynb` we connected an agent to **one remote** MCP server. In this lab we go one step further and connect **two very different servers**:

- 📁 a **Filesystem server** that runs **on our own laptop** and reads local project files
- 🐙 a **GitHub server** that runs **on GitHub's computers** and reads code repositories

The goal is not just a working agent. The goal is a **provably grounded** agent: for every answer, we can point to the **exact tool call and the exact text** that backs it up.

## 🗺️ Steps we will follow

| # | Step | What happens |
|---|------|--------------|
| 1 | **Setup** | Install packages, load the OpenAI key and the GitHub token, check Node.js |
| 2 | **Connect to the Filesystem server** | Start a local server over **stdio** |
| 3 | **Load the tools + add a guardrail** | List the tools and keep only the **read-only** ones |
| 4 | **Build an agent + show the evidence** | Ask a question and print every tool call behind the answer |
| 5 | **Connect to the GitHub server** | Call a remote server over **HTTP**, with a token |
| 6 | **Use resources** | Give the agent read-only background information |
| 7 | **One agent, both servers** | One client, two servers, one question that needs both |
| 8 | **Optional: MCP vs direct API** | Ask the same question **without** tools and compare |
| ✅ | **Your `lab_proof.md`** | What to hand in |

> 💡 **The one rule of this lab:** a working agent isn't the goal. A **provably grounded** agent is.

## The Big Picture: One Client, Two Servers

![One client, two servers: our notebook starts the filesystem server on our laptop over stdio, and calls the GitHub server on the internet over HTTP with a token](mcp_lab_two_servers.svg)

| | 📁 Filesystem server | 🐙 GitHub server |
|---|---|---|
| **Runs where?** | On **our laptop** | On **GitHub's computers** |
| **Transport** | `stdio` | `http` |
| **How the client reaches it** | It **starts** it: `command` + `args` | It **calls** it: `url` |
| **Needs a password?** | No, but it can only see **one folder** | Yes, a **GitHub token** |
| **Needs installed** | Node.js | Nothing (just internet) |

📞 **Phone analogy, continued from `mcp_introduction`:**
- **GitHub (HTTP):** we **call a company**. They are always open; we need their **number** (URL) and our **ID card** (token).
- **Filesystem (stdio):** we **bring an assistant to our desk**. They are not there until we invite them (`command`), and we tell them which **one drawer** they may open (`sample_docs/`).

## Step 1: Setup

### What you need before you start

| What | Why | How to check |
|---|---|---|
| **OpenAI API key** | The agent's "brain" | `OPENAI_API_KEY` in your `.env` file |
| **Node.js** | The Filesystem server is a JavaScript program, started with `npx` | Run `node --version` in a terminal |
| **GitHub token** (for Step 5) | GitHub needs to know who is asking | `GITHUB_PERSONAL_ACCESS_TOKEN` in your `.env` file |

🔑 **How to create the GitHub token** (2 minutes):
1. GitHub → **Settings** → **Developer settings** → **Personal access tokens** → **Fine-grained tokens** → **Generate new token**
2. **Repository access:** *Public repositories (read-only)* is enough for this lab
3. Copy the token into your `.env` file: `GITHUB_PERSONAL_ACCESS_TOKEN=github_pat_...`

> ⚠️ **Never** paste a token into a notebook cell or commit it to Git. It lives in `.env`, and `.env` stays on your laptop.

In [ ]:
%pip install -qU langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv "pydantic>=2,<3"

In [ ]:
import os
import shutil
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please add it to your .env file.")

GITHUB_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

print("OpenAI key:   ✅ loaded")
print("Node.js npx: ", "✅ found" if shutil.which("npx") else "❌ NOT found: install Node.js from nodejs.org")
print("GitHub token:", "✅ loaded" if GITHUB_TOKEN else "⚠️ missing: Steps 5 and 7 will skip GitHub")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Step 2: Connect to the Filesystem Server (stdio)

The Filesystem server is the official reference server [`@modelcontextprotocol/server-filesystem`](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem). We point it at **one folder**, [`sample_docs/`](./sample_docs), which holds two small files about a fictional client project, *Project Atlas*. You can open them yourself to check every answer the agent gives.

There is **no URL** this time: nobody hosts this server. Our client **starts it** on our laptop, exactly as if we typed this in a terminal:

```
npx -y @modelcontextprotocol/server-filesystem sample_docs
```

| Part | Meaning |
|---|---|
| `npx` | Node.js tool that downloads and runs a package |
| `-y` | "Yes, download it if needed" (otherwise `npx` waits for an answer nobody can type) |
| `@modelcontextprotocol/server-filesystem` | The server package |
| `SAMPLE_DOCS_DIR` | The **only folder** the server may touch: our security boundary |

⚙️ **One technical fix first (Windows + Jupyter only).** To start the server, the client needs a place to send the server's error messages. Inside Jupyter on Windows that place doesn't exist, so the connection fails. The cell below sends those messages to "nowhere" instead. This is plumbing, not MCP: you can run it and move on.

In [ ]:
import sys

if sys.platform == "win32":
    import mcp.client.stdio as _stdio
    _stdio.stdio_client.__wrapped__.__defaults__ = (open(os.devnull, "w"),)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

SAMPLE_DOCS_DIR = os.path.join(os.getcwd(), "sample_docs")

filesystem_client = MultiServerMCPClient({
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", SAMPLE_DOCS_DIR],
    }
})

print("Filesystem client configured")
print(f"Allowed folder: {SAMPLE_DOCS_DIR}")

> 💡 **Remember from `mcp_introduction`:** this cell only **saves the contact**. Nothing has started yet. The server is started the first time we call `get_tools()`. The first run can take ~20 seconds, because `npx` downloads the package.

## Step 3: Load the Tools + Add a Guardrail

`get_tools()` starts the server, asks *"what tools do you have?"*, and turns each one into a LangChain tool. Each tool also carries a **hint** from the server: `readOnlyHint` tells us whether the tool only **reads**, or can also **change** files.

In [ ]:
fs_tools = await filesystem_client.get_tools()

print(f"Loaded {len(fs_tools)} tools from the Filesystem server:\n")
for tool in fs_tools:
    read_only = tool.metadata.get("readOnlyHint", False)
    print(f"  {'📖 read ' if read_only else '✏️ WRITE'}  {tool.name}")

### 🛡️ Guardrail: keep only the read-only tools

Look at the list: some tools can **write**, **edit** or **move** files. Our agent only needs to **answer questions**, so it should never be able to change a client's documents.

💼 **Consulting rule:** give an agent the **smallest set of tools** it needs for the job. It's the same idea as not giving every employee admin rights.

In [ ]:
fs_read_tools = [tool for tool in fs_tools if tool.metadata.get("readOnlyHint", False)]

print(f"Kept {len(fs_read_tools)} of {len(fs_tools)} tools:")
print([tool.name for tool in fs_read_tools])

## Step 4: Build an Agent + Show the Evidence

Now we give the agent the read-only tools and ask a question that can **only** be answered from the files.

After the answer, a small helper `show_evidence()` prints **every tool call** the agent made and **what came back**. This is the evidence trail you need for your `lab_proof.md`.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def show_evidence(result):
    """Print every tool call the agent made, and what each tool sent back."""
    calls = 0
    for message in result["messages"]:
        if isinstance(message, AIMessage):
            for call in message.tool_calls:
                calls += 1
                print(f"🔧 Tool call {calls}: {call['name']}  {call['args']}")
        elif isinstance(message, ToolMessage):
            print(f"   📄 {message.name} returned: {str(message.content)[:150]}...")
    if calls == 0:
        print("⚠️ No tool calls: this answer is NOT grounded in any source!")

In [ ]:
fs_agent = create_agent(
    model=llm,
    tools=fs_read_tools,
    system_prompt=(
        f"You answer questions about client projects. The project files are in this folder: {SAMPLE_DOCS_DIR}. "
        "Start by listing that folder, then read the files you need. "
        "Always look the answer up in the files, never guess, and name the file your answer comes from."
    ),
)

question = "Who is the engagement lead for Project Atlas, and what is the target go-live date?"
print(f"Question: {question}\n")

fs_result = await fs_agent.ainvoke({"messages": [HumanMessage(content=question)]})

print("=" * 60)
show_evidence(fs_result)
print("=" * 60)
print(f"\nFinal answer:\n{fs_result['messages'][-1].content}")

### 🔍 How to check the answer is grounded

1. Find the 📄 line where a tool returned the engagement lead and the date
2. Open that file in [`sample_docs/`](./sample_docs) yourself and check it says the same
3. Check the final answer doesn't add anything that is **not** in the tool output

If all three are true, the answer is **grounded**: you have a receipt for it.

## Step 5: Connect to the GitHub Server (HTTP + token)

GitHub hosts an official MCP server at `https://api.githubcopilot.com/mcp/`. This time there is **nothing to start**: we just **call** it, like the docs server in `mcp_introduction`. The new part is the **`headers`**: extra information we send with every call.

| Header | Meaning |
|---|---|
| `Authorization` | Our **ID card**: `Bearer <token>`. Without it, GitHub answers *401 Unauthorized* |
| `X-MCP-Toolsets` | Which **groups** of tools we want. GitHub has many; we only ask for `repos` and `issues` |
| `X-MCP-Readonly` | `"true"` = give us **only read tools**. The same guardrail as Step 3, but now done by the **server** |

> ⚠️ The old npm package `@modelcontextprotocol/server-github` is **deprecated**. Use GitHub's official server shown here.

In [ ]:
if not GITHUB_TOKEN:
    print("⚠️ No GITHUB_PERSONAL_ACCESS_TOKEN in .env: skipping the GitHub steps")
else:
    github_client = MultiServerMCPClient({
        "github": {
            "transport": "http",
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {
                "Authorization": f"Bearer {GITHUB_TOKEN}",
                "X-MCP-Toolsets": "repos,issues",
                "X-MCP-Readonly": "true",
            },
        }
    })

    github_tools = await github_client.get_tools()

    print(f"Loaded {len(github_tools)} tools from the GitHub server:\n")
    for tool in github_tools:
        print(f"  - {tool.name}")

Now a question that can only be answered by **reading a file inside a GitHub repository**. We pick the repository of the library we've been using all along: `langchain-ai/langchain-mcp-adapters`.

In [ ]:
if GITHUB_TOKEN:
    github_agent = create_agent(
        model=llm,
        tools=github_tools,
        system_prompt=(
            "You answer questions about GitHub repositories by reading their files with your tools. "
            "Never guess. Always name the repository and file your answer comes from."
        ),
    )

    question = (
        "Read the README of the GitHub repository langchain-ai/langchain-mcp-adapters. "
        "Which MCP transports does it show in its examples?"
    )
    print(f"Question: {question}\n")

    github_result = await github_agent.ainvoke({"messages": [HumanMessage(content=question)]})

    print("=" * 60)
    show_evidence(github_result)
    print("=" * 60)
    print(f"\nFinal answer:\n{github_result['messages'][-1].content}")

<details>
<summary>🐳 <b>Alternative:</b> run the GitHub server on your own laptop with Docker (stdio)</summary>

Some clients don't allow tools to call outside services. Then you can run GitHub's server **locally**, with Docker. It becomes a **stdio** server, like the Filesystem one, and the token goes in `env` instead of `headers`:

```python
"github": {
    "transport": "stdio",
    "command": "docker",
    "args": ["run", "-i", "--rm", "-e", "GITHUB_PERSONAL_ACCESS_TOKEN", "ghcr.io/github/github-mcp-server"],
    "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": GITHUB_TOKEN},
}
```

Same server, same tools: only the **transport** changes.
</details>

## Step 6: Use Resources

| | 🔧 Tools | 📄 Resources |
|---|---|---|
| **What** | Actions: *"do something"* | Read-only data: *"know something"* |
| **Who decides** | The **agent** decides when to call them | **Our code** fetches them and gives them to the agent |
| **Example** | `read_text_file`, `get_file_contents` | A guide, a config file, a policy document |

⚠️ **Not every server offers resources.** Let's check the Filesystem server:

In [ ]:
try:
    fs_resources = await filesystem_client.get_resources()
    print(f"Filesystem server offers {len(fs_resources)} resources")
except Exception:
    print("❌ The Filesystem server offers NO resources (it only has tools)")

So for resources we go back to a server we already know: the **LangChain docs server** from `mcp_introduction`, which offers **1 resource**: a LangChain "cheat sheet".

This time we **don't** give the agent any tools. We put the resource straight into the **system prompt** as background knowledge, so the agent can answer without calling anything.

In [ ]:
docs_client = MultiServerMCPClient({
    "langchain-docs": {"transport": "http", "url": "https://docs.langchain.com/mcp"}
})

docs_resources = await docs_client.get_resources()
langchain_guide = docs_resources[0].as_string()
print(f"Loaded 1 resource: {len(langchain_guide)} characters of background knowledge\n")

guide_agent = create_agent(
    model=llm,
    tools=[],   # no tools: the knowledge comes from the resource
    system_prompt=(
        "Answer ONLY using the LangChain guide below. If the answer is not in the guide, say so.\n\n"
        f"--- LANGCHAIN GUIDE ---\n{langchain_guide}"
    ),
)

question = "What do I need to add to an agent so that it remembers the conversation between calls?"
guide_result = await guide_agent.ainvoke({"messages": [HumanMessage(content=question)]})

print(f"Question: {question}")
print(f"Answer:   {guide_result['messages'][-1].content}")
print(f"\nEvidence: does the resource mention 'checkpointer'? {'checkpointer' in langchain_guide}")

> ⚠️ **This resource is live:** LangChain can change it at any time. If one day the agent answers *"this is not in the guide"*, check the evidence line. If the word is gone from the resource, the agent is doing exactly the right thing: it refuses to answer beyond its source. **That is grounding working.**

> 💡 **Tools vs resources in one line:** with **tools**, the agent goes and **fetches** the information when it needs it. With **resources**, **we hand it over** before the conversation starts.

## Step 7: One Agent, Both Servers

This is where the **"Multi"** in `MultiServerMCPClient` pays off: one client, one dictionary, two servers. The agent gets tools from **both** and decides which ones to use.

🏷️ `tool_name_prefix=True` adds the server name in front of every tool name (`filesystem_read_text_file`, `github_get_file_contents`). Then the evidence trail shows **which server** every piece of evidence came from.

The question needs **both** servers: a constraint from our **local project notes**, and a fact from a **GitHub README**.

In [ ]:
servers = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", SAMPLE_DOCS_DIR],
    },
}
if GITHUB_TOKEN:
    servers["github"] = {
        "transport": "http",
        "url": "https://api.githubcopilot.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {GITHUB_TOKEN}",
            "X-MCP-Toolsets": "repos",
            "X-MCP-Readonly": "true",
        },
    }

combined_client = MultiServerMCPClient(servers, tool_name_prefix=True)

all_tools = await combined_client.get_tools()
read_tools = [tool for tool in all_tools if tool.metadata.get("readOnlyHint", False)]
print(f"Servers: {list(servers)}  |  read-only tools: {len(read_tools)} of {len(all_tools)}")

In [ ]:
combined_agent = create_agent(
    model=llm,
    tools=read_tools,
    system_prompt=(
        "You are a consultant's research assistant. "
        f"The local project files are in this folder: {SAMPLE_DOCS_DIR}. "
        "GitHub repositories are read with the github_ tools. "
        "Look everything up with your tools, never guess, and name every file and repository you used."
    ),
)

question = (
    "Read our Project Atlas notes: what constraint do they set about where ticket data may live? "
    "Then read the README of the GitHub repository langchain-ai/langchain-mcp-adapters: "
    "which transport would let us run an MCP server inside the client's own environment, "
    "so that data never leaves it?"
)
print(f"Question: {question}\n")

combined_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})

print("=" * 60)
show_evidence(combined_result)
print("=" * 60)
print(f"\nFinal answer:\n{combined_result['messages'][-1].content}")

🔌 **"Remember to disconnect when done"?** The lab instructions say this, but with `MultiServerMCPClient` there is nothing to disconnect: it's **stateless** (remember the phone analogy: every call hangs up by itself). The Filesystem server is also stopped automatically after every call.

## Step 8 (Optional): MCP vs Direct API

What if we ask the model **directly**, without any tools? Same model, same question as in Step 4:

In [ ]:
question = "Who is the engagement lead for Project Atlas, and what is the target go-live date?"

direct_answer = await llm.ainvoke([HumanMessage(content=question)])

print("DIRECT API (no MCP, no tools)")
print("=" * 60)
print(direct_answer.content)
print("\nEvidence trail: none. There is no tool call to check.")

### 🔍 What happened?

The model has **never seen** our project files, so it can only say *"I don't know"*, or worse, **invent** a name and a date that sound right.

| | 🔌 With MCP (Step 4) | 💬 Direct API |
|---|---|---|
| **Where the answer comes from** | A tool call that read `sample_docs/` | Only the model's training data |
| **Can we check it?** | ✅ Yes: we have the tool call and the text it returned | ❌ No evidence trail |
| **Up to date?** | ✅ Reads the files as they are **now** | ❌ Frozen at the model's training date |
| **Setup effort** | More: client, server, Node.js, token | One line of code |

> 💡 **Key takeaway:** MCP doesn't make the model **smarter**. It makes it **capable**, and gives every answer a **receipt**.

---

# ✅ Your `lab_proof.md`

Your `lab_proof.md` is the **entry point** for the reviewer. It needs 5 things, and this notebook already produced all of them:

| # | Section in `lab_proof.md` | Where to find it in this notebook |
|---|---|---|
| 1 | **The code or workflow** | Link to this notebook, plus 2-3 lines describing your path (which servers, which guardrail) |
| 2 | **One query** | The question from Step 4, 5 or 7 |
| 3 | **The evidence** | The `show_evidence()` output: the tool calls and what they returned |
| 4 | **The final output** | The agent's final answer |
| 5 | **One failure or limitation** | See the ideas below 👇 |

### 💡 Ideas for "one failure or limitation"

| Try this | What can go wrong |
|---|---|
| Ask something that is **not** in the files: *"What is Project Atlas's budget?"* | Does the agent admit it doesn't know, or **invent** a number? |
| Check Step 8's direct answer | A fluent answer with **zero evidence** |
| Ask a question where **only part** of the answer is in the tool output | The agent may **mix** real evidence with guesses: check every claim |
| Remove the read-only filter in Step 3 and look at the tools | The agent **could** write or move files: a real risk with client data |

> 🔑 **Before you submit:** no API keys or tokens in any file, and only the files for this lab.

## 🛠️ Troubleshooting

| Problem | Likely cause | Fix |
|---|---|---|
| `npx` not found / Filesystem connection fails | Node.js not installed | Install Node.js from nodejs.org, restart VS Code |
| `Connection closed` from the Filesystem server on Windows | The Step 2 fix cell was not run | Run the `sys.platform == "win32"` cell, then try again |
| `get_resources()` fails on the Filesystem server | This server offers **no** resources | Expected: see Step 6 |
| GitHub: `401 Unauthorized` | Token missing, wrong or expired | Check `GITHUB_PERSONAL_ACCESS_TOKEN` in `.env`, then restart the kernel |
| You see a `coroutine` instead of a result | Missing `await` | Put `await` in front of `get_tools()`, `get_resources()` and `ainvoke()` |
| The agent answers but `show_evidence()` shows no tool calls | The model answered from memory | Make the system prompt stricter: *"Always look it up, never guess"* |

## 📌 Key Takeaways

1. **stdio = the client starts the server** (`command` + `args`). **HTTP = the client calls the server** (`url` + `headers`).
2. **Local servers** (Filesystem) need the program installed; **remote servers** (GitHub) need a **token**.
3. **Guardrail:** keep only the tools the agent needs, using `readOnlyHint` or the server's read-only mode.
4. **Not every server has resources.** Tools are fetched **by the agent**; resources are handed over **by us**.
5. **One `MultiServerMCPClient` can hold many servers**, and `tool_name_prefix=True` shows which server each tool comes from.
6. **Grounded = traceable:** every claim in the answer should point to a tool call and its output.